# Tier-a Elo ladder — strength across the checkpoint history

Round-robin between `start` (pre-self-play 5M grow), `mid`
(2026-07-03 22:34), `final` (2026-07-04 09:04) and the scripted
`dummy` anchor. LADDER MAPS ONLY (pinned protocol), MCTS 32 sims
(training-matched), 66 games, 4-way parallel. Two Elo tables from
the same games: MATERIAL-SIGN (primary) and PURE (draws=0.5).

## Setup (once)
1. **Add-ons → Secrets → add `HF_TOKEN`** = your fine-grained
   Hugging Face token (read access to `momom2/wesnoth-tier-a`).
2. Settings: **Accelerator GPU T4 x2**, **Internet On**.
3. Save Version → Save & Run All. Expect ~4–6 h; per-game results
   stream into the log, and everything uploads to the HF repo
   (`elo/` folder) at the end.

In [ ]:
import os, torch
assert torch.cuda.is_available(), 'need GPU T4 x2'
print('torch', torch.__version__, '| vCPUs:', os.cpu_count())
!rm -rf /tmp/Wesnoth-AI
!git clone --depth 1 https://github.com/momom2/Wesnoth-AI.git /tmp/Wesnoth-AI
%cd /tmp/Wesnoth-AI

In [ ]:
# Fetch the three checkpoint players. `start` ships in the repo;
# mid/final come from the HF revision history (each 30-min upload
# was a commit).
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
import shutil, pathlib
TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
REPO = 'momom2/wesnoth-tier-a'
REVS = {'mid': '317d99b23642', 'final': '74d4828447c0'}
ck = pathlib.Path('/tmp/ckpts'); ck.mkdir(exist_ok=True)
shutil.copy('training/checkpoints/tier_a_5m.pt', ck / 'start.pt')
for name, rev in REVS.items():
    p = hf_hub_download(REPO, 'tier_a_campaign.pt', revision=rev,
                        token=TOKEN)
    shutil.copy(p, ck / f'{name}.pt')
!ls -la /tmp/ckpts/

In [ ]:
# Manifest + 4-way parallel driver. Each game is an independent
# process (the pattern that saturates the GPU); results are JSON
# files, so re-running skips finished games.
import itertools, subprocess, time, pathlib
OUT = pathlib.Path('/kaggle/working/elo_games'); OUT.mkdir(exist_ok=True)
P = {'start': '/tmp/ckpts/start.pt', 'mid': '/tmp/ckpts/mid.pt',
     'final': '/tmp/ckpts/final.pt', 'dummy': 'dummy'}
PAIRS = [('start', 'mid', 14), ('start', 'final', 14),
         ('mid', 'final', 14), ('dummy', 'start', 8),
         ('dummy', 'mid', 8), ('dummy', 'final', 8)]
jobs = []
seed = 90000
for a, b, n in PAIRS:
    for g in range(n):
        side_a = 1 if g % 2 == 0 else 2
        jobs.append((a, b, side_a, seed)); seed += 1
print(f'{len(jobs)} games scheduled')
MAXPROC = 4
running = []
for a, b, side_a, sd in jobs:
    while len(running) >= MAXPROC:
        time.sleep(10)
        running = [p for p in running if p.poll() is None]
    running.append(subprocess.Popen(
        ['python', 'tools/elo_eval_game.py', a, P[a], b, P[b],
         str(side_a), str(sd), str(OUT)],
        stdout=open(OUT / f'log_{a}_{b}_{sd}.txt', 'w'),
        stderr=subprocess.STDOUT))
    done = len(list(OUT.glob('game_*.json')))
    print(f'launched {a} vs {b} seed={sd} | done so far: {done}',
          flush=True)
for p in running:
    p.wait()
print('all games finished:', len(list(OUT.glob('game_*.json'))))

In [ ]:
# Fit + print both Elo tables; save and upload everything to HF.
!python tools/elo_collect.py /kaggle/working/elo_games \
    --anchor dummy --save-json /kaggle/working/elo_results.json
from huggingface_hub import HfApi
api = HfApi(token=TOKEN)
api.upload_file(path_or_fileobj='/kaggle/working/elo_results.json',
                path_in_repo='elo/elo_results.json', repo_id=REPO)
api.upload_folder(folder_path='/kaggle/working/elo_games',
                  path_in_repo='elo/games', repo_id=REPO,
                  allow_patterns=['game_*.json'])
print('uploaded to HF: elo/elo_results.json + elo/games/')